# Homework 4: Prefix Tuning for Table-to-Text Generation

## 1. Task

The task is **table-to-text generation** for the E2E restaurant dataset. Each input is a noisy meaning-representation line that lists attributes such as restaurant name, area, price range, rating, cuisine, or nearby landmark. The goal is to generate a fluent English sentence that verbalizes the important facts from that table.

**Expected input:** one line such as:

`name : Alimentum | area : city centre | family friendly : no | near : Burger King`

**Expected output:** one English sentence describing that restaurant, for example a sentence saying that Alimentum is in the city centre, is near Burger King, and is not family friendly.

The challenge is not only to generate grammatical text, but also to preserve the structured information in the table and avoid generic or repetitive language.

## 2. Method

The method uses `distilgpt2` as the base causal language model and applies **prefix tuning** with the Hugging Face PEFT library. Instead of fine-tuning all model parameters, a small set of trainable prefix parameters is attached to the frozen base model.

The final system has three main parts:

1. Build a prefix-tuned `distilgpt2` model with `PrefixTuningConfig`.
2. Train the prefix parameters on the provided `data/train.txt.gz` table-to-text data.
3. Decode with deterministic beam search and a short generation limit so the output stays concise.

The best final configuration is:

- base model: `distilgpt2`
- prefix tuning with PEFT
- prefix projection: enabled
- virtual tokens: 10
- deterministic decoding: `do_sample=False`, `num_beams=5`
- maximum generation length: `max_new_tokens=30`
- local training data loaded from `data/train.txt.gz`

Random seeds were fixed for Python and PyTorch so that the experiments were comparable across runs.

## 3. Results

### 3.1 Quantitative Results

The main comparison is between the baseline default solution and the final prefix-tuned system.

| System | `small.out` | `dev.out` |
|---|---:|---:|
| Baseline default solution | 1.5573 | 0.0000 |
| Final prefix-tuned system | **40.1003** | **36.4258** |

This is a very large improvement over the default system. The baseline mostly produces generic, off-task language-model text, while the final system generates restaurant descriptions that are much closer to the E2E references.

A sentence-level comparison on `small.txt` using `answer/analysis.py` shows the same trend:

- average sentence BLEU: `1.29 -> 40.10`
- average BLEU delta: `+38.81`
- improved / worse / tied examples: `10 / 0 / 0`
- average repeated bigrams: `16.90 -> 10.70`

These numbers show that the final model improves both content accuracy and output quality, while also reducing noisy repetition.

### 3.2 Qualitative Results

**Example 1**

- Input: `name : Alimentum | area : city centre | family friendly : no`
- Reference: `Alimentum is not a family-friendly place in the city centre.`
- Baseline output: `I've been working on a project for a while now ...`
- Final output: `Alimentum is a family friendly restaurant that serves food in the city centre . It is located in the city centre . It is located in the city`

The baseline output is unrelated to the task. The final system at least identifies the restaurant and city-centre location correctly, although it still misses the negative family-friendly attribute and repeats the location.

**Example 2**

- Input: `name : Aromi | Type : coffee shop | food : Chinese | customer rating : 1 out of 5 | area : riverside | family friendly : yes`
- Baseline output: generic blog-like text unrelated to restaurants
- Final output: `Aromi is a coffee shop that serves Chinese food in the riverside area . It has a customer rating of 1 out of 5 . It is`

Here the final system captures the name, restaurant type, cuisine, area, and rating. The sentence still ends abruptly, but it is much closer to the reference style than the baseline.

**Example 3**

- Earlier tuned output with long decoding: `Alimentum is located near Burger King in the city centre near Burger King in the city centre near Burger King ...`
- Final tuned output with shorter decoding: shorter and more focused restaurant descriptions

This example illustrates one of the main improvements from the later decoding experiments: reducing the maximum generation length removed repeated trailing phrases and improved BLEU substantially.

## 4. Alternative Methods Tried

The development process followed a sequence of controlled experiments, changing one part at a time.

### 4.1 Implementation Progression

| Step | Change | `small.out` | `dev.out` |
|---|---|---:|---:|
| Implement 1-4 | end-to-end prefix-tuning pipeline | 15.3033 | 16.9456 |
| Implement 5 | deterministic beam search | 15.3033 | 16.9456 |
| Implement 6 | prefix projection | 16.5514 | 22.2689 |
| Implement 7 | virtual tokens = 10 | 20.3171 | 22.6765 |
| Implement 8 | `max_new_tokens=40` | 28.5453 | 28.1297 |
| Implement 9 | `max_new_tokens=35` | 32.4642 | 31.4342 |
| Implement 10 | `max_new_tokens=30` | **40.1003** | **36.4258** |

This table shows that the largest gains came from the later decoding-length experiments, while the earlier implementation steps were necessary for making the tuned model work at all.

### 4.2 Prefix-Tuning Infrastructure

The first group of edits built the prefix-tuning pipeline itself:

- attach a PEFT prefix adapter to `distilgpt2`
- add the training loop
- save and reload the trained adapter
- switch from the baseline path to the trained tuned-model path

These steps were structurally necessary. On their own they did not improve BLEU much, but together they enabled the first substantial improvement by making the system actually train and decode with the tuned model.

### 4.3 Deterministic Decoding

Sampling was removed and deterministic beam search was used instead (`do_sample=False`, `num_beams=5`). This did not create a large score jump by itself, but it made later experiments fair and stable.

### 4.4 Prefix Projection

Enabling prefix projection improved `dev.out` noticeably. This suggests that the projected prefix representation gave the tuned prompt more expressive power.

### 4.5 Virtual Tokens

Increasing virtual tokens from 5 to 10 improved both `small.out` and `dev.out`. The larger prefix seems to help the model encode more of the structured restaurant information.

### 4.6 Decoding Length Experiments

The most successful sequence of experiments changed `max_new_tokens`:

- `50 -> 40` improved BLEU a lot
- `40 -> 35` improved BLEU again
- `35 -> 30` gave the best final scores

This indicates that the task benefits from short, compact outputs. Long decoding often caused repetition, extra phrases, and drift away from the concise E2E reference style.

### 4.7 Unsuccessful Alternatives

Several alternatives were tested but not kept:

- increasing beam width from 5 to 8 reduced BLEU from `small.out = 20.3171`, `dev.out = 22.6765` to `small.out = 16.9281`, `dev.out = 21.5486`
- adding `no_repeat_ngram_size=2` reduced BLEU to `small.out = 18.6623`, `dev.out = 18.6546`

These failures are useful because they show that stronger decoding constraints are not automatically better. For this task, output length control was more effective than wider search or explicit repetition blocking.

## 5. Conclusion

The default `distilgpt2` baseline performs very poorly on this table-to-text task because it generates generic language rather than restaurant descriptions grounded in the input table. Prefix tuning makes it possible to adapt the frozen language model to the task with a relatively small number of trainable parameters.

The biggest gains came from three factors:

- enabling the full end-to-end prefix-tuned training and inference path
- increasing prefix capacity with projection and 10 virtual tokens
- shortening decoding length so the output stays concise and avoids repetition

The final system reaches `40.1003` on `small.out` and `36.4258` on `dev.out`, which is a large improvement over the baseline and shows that prefix tuning is effective for noisy table-to-text generation.

## 6. References

1. Li, Xiang Lisa, and Percy Liang. **Prefix-Tuning: Optimizing Continuous Prompts for Generation.** ACL-IJCNLP 2021.

2. Novikova, Jekaterina, Ondrej Dusek, and Verena Rieser. **The E2E Dataset: New Challenges For End-to-End Generation.** SIGDIAL 2017.

3. Hugging Face PEFT documentation: prefix tuning and parameter-efficient fine-tuning for transformer models.

4. Hugging Face `e2e_nlg` dataset / cleaned E2E restaurant generation data used in the homework scaffold.